MATH2603 Group Project

Project Overview: California Housing Price Prediction using Neural Networks

本项目的目标是使用 California Housing 数据集构建一个房价的神经网络预测模型。
项目首先建立一个 Baseline 神经网络模型，仅使用原始数据特征进行预测。
随后通过构建 neighbourhood-based features（邻域特征），利用地理邻近关系kNN
提取额外信息，从而构建 Enhanced Model，提升了预测性能。

注：本项目运行时间较长约为15到55分钟，与配置环境和电脑性能相关。去除运行参数择优过程代码可以大幅降低运行时间，可直接运行第一第二第四和第十一个代码块

代码中的结果可视化
   - 绘制 Validation MSE for Different Architectures and Activations
   - 绘制 Validation MSE for Feature Combinations
   - 绘制 Validation MSE vs Number of Neighbours (Single Neighbourhood Features)
   - 绘制 Train vs Test MSE — Model 1 and Model 2
   - 绘制 Approximate Learning Curves — Overfitting / Underfitting Check\n
   - 绘制 Absolute Error vs Actual Price

PART ONE

1. 数据导入与预处理
   - 加载 California Housing 数据集
   - 处理缺失值
   - 数据标准化
   - 划分训练集、验证集和测试集

In [1]:
#数据导入及处理缺失值
#Data import and handling of missing values
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

# 加载数据
# load data
data = fetch_california_housing()

X = data.data          # 特征值eigenvalue
y = data.target        # 目标值（房价）Target value (house price)

feature_names = data.feature_names

# Transfer to DataFrame（推荐，方便后续操作）
X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name="MedHouseVal")

print(X.head())
print(y.head())
print("Shape of X:", X.shape)

# ===== 处理缺失值 =====
#Handling Missing Values
print("Missing values before cleaning:")
print(X.isnull().sum())

# 删除含缺失值的行（如果有）
# Delete rows with missing values (if any).
X = X.dropna()
y = y.loc[X.index]   # 保持 X 和 y 对齐 Align X and Y

print("Shape after removing missing values:", X.shape)

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  
0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: MedHouseVal, dtype: float64
Shape of X: (20640, 8)
Missing values before cleaning:
MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int64
Shape after removing missing values: (20640, 8)


In [2]:
#Baseline Modal
#Data preprocessing and division of training，validation and test set
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# 先从X特征值，y房价 中分出 test 和 train_val两个数据集
# First, separate the "X feature values" and "y house prices" into two datasets: "test" and "train_val".
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=26
)
# X_train_val → 80% 的特征，X_test → 20% 的特征，y_train_val → 80% 的标签，y_test → 20% 的标签
# X_train_val → 80% of the features, X_test → 20% of the features, 
# y_train_val → 80% of the labels, y_test → 20% of the labels

# 从X_train_val, y_train_val里再分 validation（分成X_train, X_val, y_train, y_val）
# From X_train_val and y_train_val, further divide into validation (resulting in X_train, X_val, y_train, y_val)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1, random_state=26
)#这里的0.1是train_val80%里的10%
#Here, 0.1 refers to the 10% within the "train_val 80%" portion.


# 在标准化前，先保存 raw（未标准化）版本：后面kNN需要用原始经纬度
# Before standardization, save the raw (unstandardized) version first: 
# the kNN algorithm later will require the original latitude and longitude values.
X_train_raw = X_train.copy()
X_val_raw   = X_val.copy()
X_test_raw  = X_test.copy()

# scale (fit only on train) and then convert it back to a DataFrame#标准化（只在训练集）并转回DataFrame
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns,index=X_train_raw.index)
X_val   = pd.DataFrame(scaler.transform(X_val), columns=X.columns,index=X_val_raw.index)
X_test  = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test_raw.index)
#scaler.transform 标准化 test 数据 means to Standardized test data
#index=X_test_raw.index Pandas 会默认生成新的 0~n 索引 Pandas will automatically generate new indices ranging from 0 to n.
#Add index=...，这样索引不会乱（对齐 y / 后面 debug 更稳）so the index won't be disordered (it will be more stable when aligning y / and then the debug part).

PART TWO

Baseline Neural Network
   - 构建前馈神经网络 (Feedforward Neural Network)
   - 尝试不同隐藏层结构与激活函数
   - 通过Validation选择最优模型参数
   - 合并Train & Validation 数据集选择最优隐藏层结构与激活函数训练baseline_final_model模型


In [ ]:
#寻找最优隐藏层与激活函数
#Search for the optimal hidden layer and activation function
import matplotlib.pyplot as plt
# 1) Define architectures and activation functions
#  定义不同的神经网络结构和激活函数
# ============================================================

# Each tuple represents the number of neurons in hidden layers.
# Example:
# (64,32) means two hidden layers with 64 and 32 neurons.
# 每个元组表示隐藏层结构。
# 例如：
# (64,32) 表示两个隐藏层，分别有 64 和 32 个神经元。
architectures = [
    (32,),
    (64,),
    (64,32),
    (128,32),
    (128,64),
    (128,64,32)
]

# Activation functions to test.
# tanh  → nonlinear symmetric activation
# relu  → widely used deep learning activation
# logistic → sigmoid nonlinear activation
# 要测试的激活函数：
# tanh → 对称的非线性函数
# relu → 深度学习中常用激活函数
# logistic → Sigmoid 非线性激活函数
activations = ['tanh', 'relu', 'logistic']


# ============================================================
# 2) Train models and collect Validation MSE
# 训练模型并记录验证集 MSE
# ============================================================
arch_search_results = []

for arch in architectures:
    for act in activations:

        # Create a neural network model with the current architecture and activation function.
        # 创建一个神经网络模型，使用当前的隐藏层结构和激活函数
        arch_model = MLPRegressor(
            hidden_layer_sizes=arch,
            activation=act,
            solver='adam',
            learning_rate_init=1e-3,
            max_iter=800,
            early_stopping=False,
            random_state=26
        )

        # Train the model on the training dataset.
        # 在训练集上训练模型
        arch_model.fit(X_train, y_train)

        # Predict house prices on the validation set
        # 在验证集上进行预测
        arch_val_pred = arch_model.predict(X_val)
        # Calculate the Mean Squared Error (MSE) on the validation set
        # 计算验证集上的均方误差（MSE）
        arch_val_mse = mean_squared_error(y_val, arch_val_pred) 

        arch_search_results.append({
            'Architecture': str(arch),
            'Activation': act,
            'Validation_MSE': arch_val_mse
        })

        print(f"Architecture: {arch}, Activation: {act}, Validation MSE: {arch_val_mse:.4f}")

# ============================================================
# 3) Convert results to DataFrame
# 将结果转换为表格
# Convert the results list into a pandas DataFrame
# and reshape it using pivot for easier visualization.
# 将实验结果转换为 DataFrame，
# 再使用 pivot 变成适合绘图的格式。

results_df = pd.DataFrame(arch_search_results)

pivot_table = results_df.pivot(
    index='Architecture',
    columns='Activation',
    values='Validation_MSE'
)

# -------------------------------------------------
# Arrange the architecture according to the complexity of the hidden layers
# 按隐藏层的复杂程度进行排列架构
# -------------------------------------------------

order = ['(32,)', '(64,)', '(64, 32)', '(128, 32)','(128, 64)', '(128, 64, 32)']
pivot_table = pivot_table.reindex(order)

print("\n=== Validation MSE Table ===")
print(pivot_table)


# ============================================================
# 4) Plot grouped bar chart
#  绘制分组柱状图
# Create a grouped bar chart showing Validation MSE for each
# architecture and activation combination.
# 绘制分组柱状图，
# 展示不同隐藏层结构和激活函数组合的 Validation MSE。

ax = pivot_table.plot(
    kind='bar',
    figsize=(10,6),
    width=0.8,
    colormap='Set2'
)

plt.ylabel("Validation MSE")
plt.xlabel("Hidden Layer Architecture")
plt.title("Validation MSE for Different Architectures and Activations")
plt.xticks(rotation=0)
plt.legend(title="Activation")




# ------------------------------------------------------------
# Add value labels above bars using patches
# 使用 patches 自动获取每个柱子的位置，避免坐标误差
# ------------------------------------------------------------
    # Place text at the center top of each bar
    # 在每个柱子的顶部中心位置标注数值

# Add value labels above bars
for p in ax.patches:
    height = p.get_height()# 获取柱子的高度（MSE值）Get bar height (MSE value)
    ax.text(
        p.get_x() + p.get_width() / 2,
        height,
        f"{height:.3f}",
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.show()

In [3]:
# baseline neural network Training

# 1) Combine original train and validation sets
#    合并原来的训练集和验证集
X_train_final_raw_base = pd.concat([X_train_raw, X_val_raw], axis=0)
y_train_final_base = pd.concat([y_train, y_val], axis=0)

# Make sure indices align
# 保证索引一致
X_train_final_raw_base = X_train_final_raw_base.sort_index()
y_train_final_base = y_train_final_base.loc[X_train_final_raw_base.index]

# Refit scaler on the final baseline training set
scaler_base_final = StandardScaler()

X_train_final_base = pd.DataFrame(
    scaler_base_final.fit_transform(X_train_final_raw_base),
    columns=X_train_final_raw_base.columns,
    index=X_train_final_raw_base.index
)

X_test_final_base = pd.DataFrame(
    scaler_base_final.transform(X_test_raw),
    columns=X_test_raw.columns,
    index=X_test_raw.index
)

# model (no internal early stopping, since you already have X_val)
baseline_final_model = MLPRegressor(
    hidden_layer_sizes=(128,32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=800,
    early_stopping=False,
    random_state=26
)

baseline_final_model.fit(X_train_final_base, y_train_final_base)

#预测集的预测
#The predictions of the prediction set
baseline_test_pred = baseline_final_model.predict(X_test_final_base)
# 训练集预测
#Training set prediction
baseline_train_pred = baseline_final_model.predict(X_train_final_base)

# 计算训练误差
# Calculate training error
baseline_train_mse = mean_squared_error(y_train_final_base, baseline_train_pred)
baseline_test_mse_final = mean_squared_error(y_test, baseline_test_pred)

print("Final Baseline Train MSE:", baseline_train_mse)#加入是为了判断是不是欠拟合或者过拟合Joining is to determine whether it is underfitting or overfitting.
print("Final Baseline Test MSE:", baseline_test_mse_final)

Final Baseline Train MSE: 0.23981523899485185
Final Baseline Test MSE: 0.24851459335241777


PART THREE

Enhanced Model (Neighbourhood Features)
   - 使用 k-nearest neighbours (kNN) 构建邻域关系
   - 从邻居样本中提取邻居票价的统计特征
   - 研究不同数量的邻居特征对Val MSE的影响
   - 研究相同邻居数量特征与不同k值数量对Val MSE的影响
   - 将最优邻域特征加入原始特征重新训练模型
   - 合并Train & Validation 数据集选择最优邻居特征与k值训练model_enh_final模型

In [4]:
# Model 2: Add neighbourhood-based features

from sklearn.neighbors import NearestNeighbors

K = 7  # Number of nearest neighbours used to construct neighbourhood features


#   Fit kNN using training coordinates only, this ensures neighbourhood features are constructed 
#   solely from training data and avoids data leakage.
knn = NearestNeighbors(n_neighbors=K + 1)
knn.fit(X_train_raw[['Latitude', 'Longitude']])

# Align y_train with X_train_raw so neighbour indices correctly map to prices
y_train_aligned = y_train.loc[X_train_raw.index].to_numpy()



# Define a function to compute neighbourhood price statistics
#这部分是用来计算第一个图所需要的数据
def neighbour_summary_feature(query_df, summary="mean", exclude_self=False):
    # Find nearest neighbours based on geographic coordinates
    _, inds = knn.kneighbors(query_df[['Latitude', 'Longitude']])

    values = []

    # Loop through each sample and compute the summary statistic
    for row_i, idxs in enumerate(inds):

        if exclude_self:
            # Remove the sample itself if it appears in neighbours
            idxs = idxs[idxs != row_i]

            # Keep exactly K neighbours
            idxs = idxs[:K]

        else:
            # Validation/test samples are not part of training set
            # so we directly take the first K neighbours
            idxs = idxs[:K]

        # Retrieve neighbour prices
        neigh_prices = y_train_aligned[idxs]

        if summary == "mean":
            values.append(np.mean(neigh_prices))

        elif summary == "median":
            values.append(np.median(neigh_prices))

        elif summary == "min":
            values.append(np.min(neigh_prices))

        elif summary == "max":
            values.append(np.max(neigh_prices))

        elif summary == "range":
            values.append(np.max(neigh_prices) - np.min(neigh_prices))

        else:
            raise ValueError("summary must be one of: mean, median, min, max, range")

    return np.array(values)



# Neighbourhood mean price / 邻域平均房价
neigh_price_mean_train = neighbour_summary_feature(X_train_raw, summary="mean", exclude_self=True)
neigh_price_mean_val   = neighbour_summary_feature(X_val_raw, summary="mean", exclude_self=False)
neigh_price_mean_test  = neighbour_summary_feature(X_test_raw, summary="mean", exclude_self=False)

# Neighbourhood median price / 邻域房价中位数
neigh_price_median_train = neighbour_summary_feature(X_train_raw, summary="median", exclude_self=True)
neigh_price_median_val   = neighbour_summary_feature(X_val_raw, summary="median", exclude_self=False)
neigh_price_median_test  = neighbour_summary_feature(X_test_raw, summary="median", exclude_self=False)

# Neighbourhood minimum price / 邻域最低房价
neigh_price_min_train = neighbour_summary_feature(X_train_raw, summary="min", exclude_self=True)
neigh_price_min_val   = neighbour_summary_feature(X_val_raw, summary="min", exclude_self=False)
neigh_price_min_test  = neighbour_summary_feature(X_test_raw, summary="min", exclude_self=False)

# Neighbourhood maximum price / 邻域最高房价
neigh_price_max_train = neighbour_summary_feature(X_train_raw, summary="max", exclude_self=True)
neigh_price_max_val   = neighbour_summary_feature(X_val_raw, summary="max", exclude_self=False)
neigh_price_max_test  = neighbour_summary_feature(X_test_raw, summary="max", exclude_self=False)

# Neighbourhood price range / 邻域房价极差
neigh_price_range_train = neighbour_summary_feature(X_train_raw, summary="range", exclude_self=True)
neigh_price_range_val   = neighbour_summary_feature(X_val_raw, summary="range", exclude_self=False)
neigh_price_range_test  = neighbour_summary_feature(X_test_raw, summary="range", exclude_self=False)




### 用于计算合并之前的baseline validation MSE的函数
#后面用到的baseline model都是这个，就不用繁琐的每次都写了
scaler_base = StandardScaler()

X_train_base = scaler_base.fit_transform(X_train_raw)
X_val_base = scaler_base.transform(X_val_raw)

baseline_model = MLPRegressor(
    hidden_layer_sizes=(128, 32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=800,
    early_stopping=False,
    random_state=26
)

#计算后续需要的baseline validation MSE
baseline_model.fit(X_train_base, y_train)
baseline_val_pred = baseline_model.predict(X_val_base)
baseline_val_mse = mean_squared_error(y_val, baseline_val_pred)

In [5]:
# ============================================================
# Evaluate combinations of neighbourhood-based features
# This section tests multiple feature combinations
# (1-feature, 2-feature, and 3-feature combinations)
# ============================================================

from itertools import combinations
import pandas as pd


# ------------------------------------------------------------
# Store neighbourhood feature arrays in dictionaries
# Each key represents the feature name
# Each value is the corresponding feature array
# ------------------------------------------------------------
feature_pool_train = {
    "Mean": neigh_price_mean_train,
    "Median": neigh_price_median_train,
    "Min": neigh_price_min_train,
    "Max": neigh_price_max_train,
    "Range": neigh_price_range_train
}

feature_pool_val = {
    "Mean": neigh_price_mean_val,
    "Median": neigh_price_median_val,
    "Min": neigh_price_min_val,
    "Max": neigh_price_max_val,
    "Range": neigh_price_range_val
}



# ------------------------------------------------------------
# Train and evaluate a model with multiple new features
# ------------------------------------------------------------
def run_multi_features(feature_dict_train, feature_dict_val):

    # Copy original datasets to avoid modifying them
    Xtr = X_train_raw.copy()
    Xva = X_val_raw.copy()

    
    for col, arr in feature_dict_train.items():
        Xtr[col] = arr

    for col, arr in feature_dict_val.items():
        Xva[col] = arr

    
    sc = StandardScaler()

    Xtr_s = sc.fit_transform(Xtr)
    Xva_s = sc.transform(Xva)


    baseline_model.fit(Xtr_s, y_train)

    # Evaluate model performance
    baseline_val_pred  = baseline_model.predict(Xva_s)

    return mean_squared_error(y_val, baseline_val_pred)


# ------------------------------------------------------------
# Evaluate different feature combinations
# ------------------------------------------------------------
def evaluate_feature_combinations(
    feature_pool_train,
    feature_pool_val,
    combo_sizes=None,
    selected_combos=None
):

    feature_names = list(feature_pool_train.keys())
    combo_neighbour_results = []

    # Determine which combinations to evaluate
    if selected_combos is not None:
        combos_to_run = selected_combos

    else:
        # If no specific combinations are given,
        # generate combinations automatically
        if combo_sizes is None:
            combo_sizes = list(range(1, len(feature_names) + 1))

        combos_to_run = []

        # Generate combinations of size k
        for k in combo_sizes:
            combos_to_run.extend(combinations(feature_names, k))

    # ------------------------------------------------------------
    # Run experiments for each feature combination
    #
    # 对每个特征组合运行实验
    # ------------------------------------------------------------
    for combo in combos_to_run:

        combo = tuple(combo)

        # Extract the corresponding feature arrays
        feature_dict_train = {name: feature_pool_train[name] for name in combo}
        feature_dict_val   = {name: feature_pool_val[name] for name in combo}

        # Train and evaluate the model
        val_mse = run_multi_features(
            feature_dict_train,
            feature_dict_val,
        )

        # Store experiment results
        combo_neighbour_results.append({
            "combo_name": " + ".join(combo),
            "num_features": len(combo),
            "val_mse": val_mse
        })

    # ------------------------------------------------------------
    # Convert results to DataFrame
    # and sort by number of features and validation MSE
    # ------------------------------------------------------------
    return pd.DataFrame(combo_neighbour_results).sort_values(
        by=["num_features", "val_mse"],
        ascending=[True, True]
    ).reset_index(drop=True)


# ------------------------------------------------------------
# Run the experiment for
# 1-feature, 2-feature, and 3-feature combinations
# ------------------------------------------------------------
combo_results_df = evaluate_feature_combinations(
    feature_pool_train,
    feature_pool_val,
    combo_sizes=[1, 2, 3]
)

# Display the results
# 显示实验结果
display(combo_results_df)

,combo_name,num_features,val_mse
0,Mean,1,0.211151
1,Median,1,0.215764
2,Max,1,0.231737
3,Min,1,0.252567
4,Range,1,0.283184
5,Median + Range,2,0.217884
6,Median + Max,2,0.219297
7,Mean + Max,2,0.219599
8,Median + Min,2,0.220907
9,Mean + Median,2,0.222763


In [ ]:
# ============================================================
# Plot: Validation MSE vs Number of Neighbourhood Features
# ============================================================

import matplotlib.pyplot as plt
import numpy as np


# ------------------------------------------------------------
# Compute summary statistics for each feature count
# (minimum, average, and maximum validation MSE)
# ------------------------------------------------------------
summary_df = (
    combo_results_df
    .groupby("num_features")["val_mse"]   # Group by number of features 
    .agg(["min", "mean", "max"])          # Compute min / mean / max 
    .reset_index()                       # Reset index for plotting 
)

# ------------------------------------------------------------
# Plot scatter points for all feature combinations
# ------------------------------------------------------------
plt.figure(figsize=(10,6))

np.random.seed(26)

plt.scatter(
    combo_results_df["num_features"],   
    combo_results_df["val_mse"],        
    alpha=0.7,
    label="All combinations"
)


# ------------------------------------------------------------
# Plot trend lines
# Show min / average / max performance
# ------------------------------------------------------------
plt.plot(
    summary_df["num_features"],
    summary_df["min"],
    marker="o",
    label="Minimum"
)

plt.plot(
    summary_df["num_features"],
    summary_df["mean"],
    marker="o",
    label="Average"
)

plt.plot(
    summary_df["num_features"],
    summary_df["max"],
    marker="o",
    label="Maximum"
)


# ------------------------------------------------------------
# Plot baseline reference line
# ------------------------------------------------------------
plt.axhline(
    y=baseline_val_mse,
    linestyle="--",
    label="Baseline model"
)


# ------------------------------------------------------------
# Figure formatting
# ------------------------------------------------------------
plt.xlabel("Number of Neighbourhood Features")   # x轴标签
plt.ylabel("Validation MSE")                     # y轴标签
plt.title("Validation MSE for Feature Combinations")  # 图标题

# Show all tested feature counts on x-axis
plt.xticks(sorted(combo_results_df["num_features"].unique()))

plt.legend()                 
plt.grid(True, alpha=0.3)    

plt.show()

In [ ]:
# ============================================================
# Generate results_k_feature_df for Figure 2 (Validation MSE)
# All 2-feature combinations
# ============================================================

from sklearn.neighbors import NearestNeighbors
from itertools import combinations

k_values = [3, 5, 7, 10, 15]

def compute_features_for_k(K):
    knn = NearestNeighbors(n_neighbors=K + 1)
    knn.fit(X_train_raw[['Latitude', 'Longitude']])

    y_train_aligned = y_train.loc[X_train_raw.index].to_numpy()

    # Compute neighbour indices once
    _, idx_train = knn.kneighbors(X_train_raw[['Latitude', 'Longitude']])
    _, idx_val = knn.kneighbors(X_val_raw[['Latitude', 'Longitude']])

    # Remove self for training set
    idx_train = idx_train[:, 1:K+1]
    idx_val = idx_val[:, :K]

    # Convert neighbour indices to price matrices
    train_prices = y_train_aligned[idx_train]
    val_prices = y_train_aligned[idx_val]

    # Vectorised summaries
    neigh_price_train_k = train_prices.mean(axis=1)
    neigh_price_val_k   = val_prices.mean(axis=1)

    neigh_price_median_train_k = np.median(train_prices, axis=1)
    neigh_price_median_val_k   = np.median(val_prices, axis=1)

    neigh_price_min_train_k = train_prices.min(axis=1)
    neigh_price_min_val_k   = val_prices.min(axis=1)

    neigh_price_max_train_k = train_prices.max(axis=1)
    neigh_price_max_val_k   = val_prices.max(axis=1)

    neigh_price_range_train_k = neigh_price_max_train_k - neigh_price_min_train_k
    neigh_price_range_val_k   = neigh_price_max_val_k - neigh_price_min_val_k

    return {
        "Mean":   (neigh_price_train_k, neigh_price_val_k),
        "Median": (neigh_price_median_train_k, neigh_price_median_val_k),
        "Min":    (neigh_price_min_train_k, neigh_price_min_val_k),
        "Max":    (neigh_price_max_train_k, neigh_price_max_val_k),
        "Range":  (neigh_price_range_train_k, neigh_price_range_val_k),
    }


def run_one_feature(feature_name, f_train, f_val):
    Xtr = X_train_raw.copy()
    Xva = X_val_raw.copy()

    # Add one neighbourhood feature
    Xtr[feature_name] = f_train
    Xva[feature_name] = f_val

    # Re-standardise after adding the new feature
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr)
    Xva_s = sc.transform(Xva)

    # Train and evaluate
    baseline_model.fit(Xtr_s, y_train)
    val_pred = baseline_model.predict(Xva_s)

    return mean_squared_error(y_val, val_pred)

k_feature_results = []

for K in k_values:
    feature_data = compute_features_for_k(K)

    for feature_name, (f_train, f_val) in feature_data.items():
        val_mse = run_one_feature(feature_name, f_train, f_val)

        k_feature_results.append({
            "k": K,
            "feature": feature_name,
            "val_mse": val_mse
        })

k_feature_results_df = pd.DataFrame(k_feature_results)

In [ ]:
# ============================================================
# Figure 2: Validation MSE vs Number of Neighbours (k)
# Single neighbourhood features
# ============================================================

plt.figure(figsize=(10, 6))

# ------------------------------------------------------------
# Plot validation MSE curves for each single feature
# ------------------------------------------------------------
for feature_name in k_feature_results_df["feature"].unique():

    # Select rows corresponding to the current feature
    subset = k_feature_results_df[
        k_feature_results_df["feature"] == feature_name
    ].sort_values("k")

    # Plot validation MSE as a function of k
    plt.plot(
        subset["k"],
        subset["val_mse"],
        marker="o",
        label=feature_name
    )

# ------------------------------------------------------------
# Plot baseline model performance as a reference line
# ------------------------------------------------------------
plt.axhline(
    y=baseline_val_mse,
    linestyle="--",
    label="Baseline model"
)

# ------------------------------------------------------------
# Figure formatting
# ------------------------------------------------------------
plt.xlabel("Number of Neighbours (k)")
plt.ylabel("Validation MSE")
plt.title("Validation MSE vs Number of Neighbours (Single Neighbourhood Features)")

# Ensure all tested k values appear on the x-axis
plt.xticks(sorted(k_feature_results_df["k"].unique()))

# Place legend outside the plot to avoid overlapping
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

# Add grid for readability
plt.grid(True, alpha=0.3)

# Display the figure
plt.show()

In [ ]:
# from sklearn.neighbors import NearestNeighbors



# ============================================================
# 1) Combine original train and validation sets
# ============================================================
X_train_final_raw = pd.concat([X_train_raw, X_val_raw], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

# Make sure indices align
X_train_final_raw = X_train_final_raw.sort_index()
y_train_final = y_train_final.loc[X_train_final_raw.index]

# ============================================================
# 2) Fit kNN on the new final training set
# ============================================================
best_K = 10
knn_final = NearestNeighbors(n_neighbors=best_K + 1)
knn_final.fit(X_train_final_raw[['Latitude', 'Longitude']])

y_train_final_aligned = y_train_final.to_numpy()

# ============================================================
# 3) Define function to compute neighbourhood summary features
# ============================================================
def neighbour_summary_feature_final(query_df, summary="mean", exclude_self=False):
    _, inds = knn_final.kneighbors(query_df[['Latitude', 'Longitude']])
    values = []

    for row_i, idxs in enumerate(inds):
        if exclude_self:
            idxs = idxs[idxs != row_i]
            idxs = idxs[:best_K]
        else:
            idxs = idxs[:best_K]

        neigh_prices = y_train_final_aligned[idxs]

        if summary == "mean":
            values.append(np.mean(neigh_prices))
        elif summary == "median":
            values.append(np.median(neigh_prices))
        elif summary == "min":
            values.append(np.min(neigh_prices))
        elif summary == "max":
            values.append(np.max(neigh_prices))
        elif summary == "range":
            values.append(np.max(neigh_prices) - np.min(neigh_prices))
        else:
            raise ValueError("summary must be one of: mean, median, min, max, range")

    return np.array(values)

# ============================================================
# 4) Compute only the selected best features
# ============================================================
feature_map = {
    "Mean": "mean",
    "Median": "median",
    "Min": "min",
    "Max": "max",
    "Range": "range"
}

best_features = ["Mean"]   # 输入最终选出来的组合

train_final_feature_values = {}
test_feature_values = {}

for feat in best_features:
    summary_name = feature_map[feat]
    train_final_feature_values[feat] = neighbour_summary_feature_final(
        X_train_final_raw, summary=summary_name, exclude_self=True
    )
    test_feature_values[feat] = neighbour_summary_feature_final(
        X_test_raw, summary=summary_name, exclude_self=False
    )


# ============================================================
# 6) Final enhanced model
# ============================================================
X_train_enh_final_raw = X_train_final_raw.copy()
X_test_enh_final_raw = X_test_raw.copy()

for feat in best_features:
    col_name = f"NeighbourPrice{feat}"
    X_train_enh_final_raw[col_name] = train_final_feature_values[feat]
    X_test_enh_final_raw[col_name] = test_feature_values[feat]

scaler_enh_final = StandardScaler()

X_train_enh_final = pd.DataFrame(
    scaler_enh_final.fit_transform(X_train_enh_final_raw),
    columns=X_train_enh_final_raw.columns,
    index=X_train_enh_final_raw.index
)

X_test_enh_final = pd.DataFrame(
    scaler_enh_final.transform(X_test_enh_final_raw),
    columns=X_test_enh_final_raw.columns,
    index=X_test_enh_final_raw.index
)

model_enh_final = MLPRegressor(
    hidden_layer_sizes=(128, 32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=800,
    early_stopping=False,
    random_state=26
)

model_enh_final.fit(X_train_enh_final, y_train_final)
test_pred_enh_final = model_enh_final.predict(X_test_enh_final)
enhanced_test_mse_final = mean_squared_error(y_test, test_pred_enh_final)

# ============================================================
# 7) Final comparison
# ============================================================
print("Final Baseline Test MSE:", baseline_test_mse_final)
print("Final Enhanced Test MSE:", enhanced_test_mse_final)
print("Delta Test MSE (Enhanced - Baseline):", enhanced_test_mse_final - baseline_test_mse_final)

PART FOUR

baseline_final_model与model_enh_final模型比较
   - 检测最终 Baseline 与 Enhanced 模型是否出现欠拟合或过拟合
   - 比较最终 Baseline 与 Enhanced 模型 MSE 的 improvement percentages
   - 研究最终 Baseline 与 Enhanced 模型在不同房价中预测准确性的影响